# 🏥 Medical Q&A System — RAG + LLM

**Tech Stack:** Python | LangChain | FAISS | HuggingFace Embeddings | Groq LLM (Llama3)

**How it works:**
1. Load medical knowledge base and split into chunks
2. Convert chunks to vector embeddings using HuggingFace
3. Store embeddings in FAISS vector store
4. User asks a question → FAISS retrieves top 3 relevant chunks
5. Retrieved context + question → LLM generates grounded answer

**Diseases Covered:** Diabetes | Hypertension | CKD | Brain Stroke | Liver Disease | Lung Cancer | Heart Disease

## Step 1 — Install Dependencies

In [2]:
!pip install groq faiss-cpu sentence-transformers langchain-community langchain-text-splitters langchain-huggingface

## Step 2 — Import Libraries

In [3]:
import os
import warnings
warnings.filterwarnings('ignore')

from groq import Groq
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

print('✅ All libraries imported successfully')

✅ All libraries imported successfully


## Step 3 — Set Groq API Key

> 🔑 Get your FREE key at: https://console.groq.com → Sign up → API Keys → Create Key

In [4]:
from transformers import pipeline

generator = pipeline("question-answering", model="deepset/roberta-base-squad2")

def ask(question):
    retrieved_docs = vectorstore.similarity_search(question, k=3)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    result = generator(question=question, context=context[:800])
    answer = result['answer']
    
    print(f'\n🔍 Question: {question}')
    print('-' * 60)
    print(f'💊 Answer:\n{answer}')
    print('\n📄 Sources used:')
    for i, doc in enumerate(retrieved_docs):
        print(f'  Chunk {i+1}: {doc.page_content[:100]}...')
    return answer

print("✅ ask() function ready!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ask() function ready!


## Step 4 — Create Medical Knowledge Base

In [5]:
medical_text = """
DIABETES
Diabetes is a chronic disease that occurs when the pancreas does not produce enough insulin or when the body cannot effectively use the insulin it produces.
Symptoms of diabetes include frequent urination, excessive thirst, unexplained weight loss, extreme fatigue, blurry vision, slow-healing cuts or bruises, and tingling or numbness in hands or feet.
Type 1 diabetes is an autoimmune condition where the immune system attacks insulin-producing cells. It requires daily insulin injections.
Type 2 diabetes is the most common form. It is managed through diet, exercise, and medication.
Risk factors for diabetes include obesity, physical inactivity, family history, age over 45, high blood pressure, and high cholesterol.
Complications include heart disease, kidney damage, eye damage, nerve damage, and foot problems.
Diagnosis is done through fasting blood glucose test, HbA1c test, or oral glucose tolerance test.

HYPERTENSION (HIGH BLOOD PRESSURE)
Hypertension is a condition where blood pressure is persistently elevated above 130/80 mmHg.
Most people have no symptoms, which is why it is called the silent killer.
Risk factors include obesity, high salt diet, smoking, excessive alcohol, stress, family history, and age.
Treatment includes lifestyle changes: reducing salt, regular exercise, weight loss, limiting alcohol, quitting smoking.
Medications include ACE inhibitors, calcium channel blockers, beta blockers, and diuretics.
Uncontrolled hypertension can lead to heart attack, stroke, kidney failure, and vision loss.

CHRONIC KIDNEY DISEASE (CKD)
Chronic kidney disease is the gradual loss of kidney function over time.
Main causes of CKD are diabetes and high blood pressure.
Symptoms include fatigue, swollen ankles, shortness of breath, blood in urine, foamy urine, and difficulty sleeping.
CKD is diagnosed through blood tests (creatinine, GFR) and urine tests (albumin).
Stages range from Stage 1 (mild) to Stage 5 (kidney failure requiring dialysis or transplant).
Treatment includes controlling blood sugar, managing blood pressure, low-protein diet, and avoiding NSAIDs.

BRAIN STROKE
A stroke occurs when blood supply to part of the brain is cut off or a blood vessel bursts.
Types include ischemic stroke (blockage, 87% of strokes) and hemorrhagic stroke (bleeding).
Risk factors include high blood pressure, diabetes, smoking, obesity, heart disease, high cholesterol, and age above 55.
Symptoms follow the FAST acronym: Face drooping, Arm weakness, Speech difficulty, Time to call emergency.
Treatment for ischemic stroke includes clot-busting drugs (tPA) given within 4.5 hours of onset.
Prevention includes controlling blood pressure, quitting smoking, regular exercise, healthy diet, and managing diabetes.

LIVER DISEASE
The liver performs over 500 functions including detoxification, protein synthesis, and digestion support.
Common liver diseases include fatty liver disease, hepatitis, cirrhosis, and liver cancer.
Symptoms include jaundice, abdominal pain, swelling, fatigue, nausea, and dark urine.
Diagnosis is done through liver function tests (LFTs), ultrasound, CT scan, and liver biopsy.
Treatment depends on the cause: antivirals for hepatitis, lifestyle changes for fatty liver, liver transplant for end-stage.

LUNG CANCER
Lung cancer is one of the most common and serious cancers worldwide.
Risk factors include smoking (85% of cases), secondhand smoke, radon gas, asbestos, air pollution, and family history.
Symptoms include persistent cough, coughing up blood, chest pain, shortness of breath, weight loss, and fatigue.
Treatment options include surgery, radiation, chemotherapy, targeted therapy, and immunotherapy.
Early detection significantly improves survival rates.

HEART DISEASE
Heart disease includes coronary artery disease, heart failure, and arrhythmias.
Symptoms include chest pain, shortness of breath, fatigue, and swelling in legs.
Risk factors include high blood pressure, high cholesterol, smoking, diabetes, obesity, and family history.
Heart attack occurs when blood flow to part of the heart is blocked.
Treatment includes medications, lifestyle changes, angioplasty, and bypass surgery.
"""

with open('medical_data.txt', 'w') as f:
    f.write(medical_text)

print('✅ Medical knowledge base saved')
print(f'Total characters: {len(medical_text)}')

✅ Medical knowledge base saved
Total characters: 4169


## Step 5 — Load and Split into Chunks

In [6]:
loader = TextLoader('medical_data.txt')
documents = loader.load()

print(f'✅ Loaded {len(documents)} document(s)')
print(f'Content length: {len(documents[0].page_content)} characters')

✅ Loaded 1 document(s)
Content length: 4169 characters


In [7]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print(f'✅ Total chunks created: {len(chunks)}')
print('\n--- Sample Chunk ---')
print(chunks[0].page_content)

✅ Total chunks created: 13

--- Sample Chunk ---
DIABETES
Diabetes is a chronic disease that occurs when the pancreas does not produce enough insulin or when the body cannot effectively use the insulin it produces.
Symptoms of diabetes include frequent urination, excessive thirst, unexplained weight loss, extreme fatigue, blurry vision, slow-healing cuts or bruises, and tingling or numbness in hands or feet.


## Step 6 — Create Vector Embeddings + Build FAISS Vector Store

In [8]:
print('Loading embedding model... (first time may take 1-2 minutes)')

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

print('✅ Embedding model loaded')

Loading embedding model... (first time may take 1-2 minutes)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded


In [9]:
print('Building FAISS vector store...')

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local('faiss_index')

print(f'✅ Vector store built with {len(chunks)} vectors')
print('✅ Saved to faiss_index/')

Building FAISS vector store...
✅ Vector store built with 13 vectors
✅ Saved to faiss_index/


## Step 7 — Test Retrieval (Without LLM)

Let's verify FAISS retrieves the correct context.

In [10]:
# Step 7 — Test Retrieval (FIXED — no retriever.invoke)
test_query = "What are the symptoms of diabetes?"

# Direct FAISS search — bypasses LangChain retriever bug on Kaggle
retrieved_docs = vectorstore.similarity_search(test_query, k=3)

print(f"Query: {test_query}")
print(f"\nRetrieved {len(retrieved_docs)} chunks:\n")
for i, doc in enumerate(retrieved_docs):
    print(f"Chunk {i+1}:\n{doc.page_content[:200]}...")
    print("-" * 50)

Query: What are the symptoms of diabetes?

Retrieved 3 chunks:

Chunk 1:
DIABETES
Diabetes is a chronic disease that occurs when the pancreas does not produce enough insulin or when the body cannot effectively use the insulin it produces.
Symptoms of diabetes include frequ...
--------------------------------------------------
Chunk 2:
Type 1 diabetes is an autoimmune condition where the immune system attacks insulin-producing cells. It requires daily insulin injections.
Type 2 diabetes is the most common form. It is managed through...
--------------------------------------------------
Chunk 3:
Diagnosis is done through fasting blood glucose test, HbA1c test, or oral glucose tolerance test....
--------------------------------------------------


## Step 8 — Define Prompt Template (Prompt Engineering)

In [11]:
# Step 8 — Prompt Template (manual, no LangChain PromptTemplate)

def build_prompt(context, question):
    return f"""You are a helpful medical assistant. Use the following medical context to answer 
the question clearly and accurately. If you don't know the answer from the context, 
say "I don't have enough information. Please consult a doctor."

Context:
{context}

Question: {question}

Answer:"""

print("✅ Prompt template defined")

✅ Prompt template defined


## Step 9 — Build Full RAG Chain

In [13]:
def ask(question):
    # Retrieve relevant chunks from FAISS
    retrieved_docs = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    # Call Groq LLM directly
    response = client.chat.completions.create(
        model="llama3-8b-8192",
        messages=[
            {
                "role": "system",
                "content": "You are a helpful medical assistant. Answer only based on the context provided. If unsure, say please consult a doctor."
            },
            {
                "role": "user", 
                "content": f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
            }
        ],
        temperature=0.2
    )
    
    answer = response.choices[0].message.content
    print(f'\n🔍 Question: {question}')
    print('-' * 60)
    print(f'💊 Answer:\n{answer}')
    print('\n📄 Sources used:')
    for i, doc in enumerate(retrieved_docs):
        print(f'  Chunk {i+1}: {doc.page_content[:100]}...')
    return answer

print('✅ ask() function ready!')

✅ ask() function ready!


## Step 10 — Ask Medical Questions

In [14]:
from transformers import pipeline as hf_pipeline

generator = hf_pipeline("question-answering", model="deepset/roberta-base-squad2")

def ask(question):
    retrieved_docs = vectorstore.similarity_search(question, k=3)
    context = "\n\n".join([doc.page_content for doc in retrieved_docs])
    
    result = generator(question=question, context=context[:800])
    answer = result['answer']
    
    print(f'\n🔍 Question: {question}')
    print('-' * 60)
    print(f'💊 Answer:\n{answer}')
    print('\n📄 Sources used:')
    for i, doc in enumerate(retrieved_docs):
        print(f'  Chunk {i+1}: {doc.page_content[:100]}...')
    return answer

print("✅ ask() ready!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaForQuestionAnswering LOAD REPORT from: deepset/roberta-base-squad2
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ ask() ready!


In [15]:
ask('What are the symptoms of diabetes?')


🔍 Question: What are the symptoms of diabetes?
------------------------------------------------------------
💊 Answer:
frequent urination, excessive thirst, unexplained weight loss

📄 Sources used:
  Chunk 1: DIABETES
Diabetes is a chronic disease that occurs when the pancreas does not produce enough insulin...
  Chunk 2: Type 1 diabetes is an autoimmune condition where the immune system attacks insulin-producing cells. ...
  Chunk 3: Diagnosis is done through fasting blood glucose test, HbA1c test, or oral glucose tolerance test....


'frequent urination, excessive thirst, unexplained weight loss'

In [16]:
ask('How is hypertension treated?')


🔍 Question: How is hypertension treated?
------------------------------------------------------------
💊 Answer:
lifestyle changes

📄 Sources used:
  Chunk 1: Medications include ACE inhibitors, calcium channel blockers, beta blockers, and diuretics.
Uncontro...
  Chunk 2: Treatment includes controlling blood sugar, managing blood pressure, low-protein diet, and avoiding ...
  Chunk 3: HYPERTENSION (HIGH BLOOD PRESSURE)
Hypertension is a condition where blood pressure is persistently ...


'lifestyle changes'

In [17]:
ask('What causes chronic kidney disease?')


🔍 Question: What causes chronic kidney disease?
------------------------------------------------------------
💊 Answer:
diabetes and high blood pressure

📄 Sources used:
  Chunk 1: CHRONIC KIDNEY DISEASE (CKD)
Chronic kidney disease is the gradual loss of kidney function over time...
  Chunk 2: Type 1 diabetes is an autoimmune condition where the immune system attacks insulin-producing cells. ...
  Chunk 3: DIABETES
Diabetes is a chronic disease that occurs when the pancreas does not produce enough insulin...


'diabetes and high blood pressure'

In [18]:
ask('What are the risk factors for brain stroke?')


🔍 Question: What are the risk factors for brain stroke?
------------------------------------------------------------
💊 Answer:
high blood pressure, diabetes, smoking, obesity, heart disease, high cholesterol

📄 Sources used:
  Chunk 1: BRAIN STROKE
A stroke occurs when blood supply to part of the brain is cut off or a blood vessel bur...
  Chunk 2: Treatment for ischemic stroke includes clot-busting drugs (tPA) given within 4.5 hours of onset.
Pre...
  Chunk 3: Medications include ACE inhibitors, calcium channel blockers, beta blockers, and diuretics.
Uncontro...


'high blood pressure, diabetes, smoking, obesity, heart disease, high cholesterol'

In [19]:
ask('How is liver disease diagnosed?')


🔍 Question: How is liver disease diagnosed?
------------------------------------------------------------
💊 Answer:
liver function tests

📄 Sources used:
  Chunk 1: LIVER DISEASE
The liver performs over 500 functions including detoxification, protein synthesis, and...
  Chunk 2: Diagnosis is done through fasting blood glucose test, HbA1c test, or oral glucose tolerance test....
  Chunk 3: Treatment depends on the cause: antivirals for hepatitis, lifestyle changes for fatty liver, liver t...


'liver function tests'

In [20]:
ask('What are the symptoms of lung cancer?')


🔍 Question: What are the symptoms of lung cancer?
------------------------------------------------------------
💊 Answer:
persistent cough, coughing up blood

📄 Sources used:
  Chunk 1: LUNG CANCER
Lung cancer is one of the most common and serious cancers worldwide.
Risk factors includ...
  Chunk 2: HEART DISEASE
Heart disease includes coronary artery disease, heart failure, and arrhythmias.
Sympto...
  Chunk 3: LIVER DISEASE
The liver performs over 500 functions including detoxification, protein synthesis, and...


'persistent cough, coughing up blood'

In [21]:
# Try your own question!
ask('What are the complications of diabetes?')


🔍 Question: What are the complications of diabetes?
------------------------------------------------------------
💊 Answer:
heart disease, kidney damage, eye damage, ner

📄 Sources used:
  Chunk 1: DIABETES
Diabetes is a chronic disease that occurs when the pancreas does not produce enough insulin...
  Chunk 2: Type 1 diabetes is an autoimmune condition where the immune system attacks insulin-producing cells. ...
  Chunk 3: Treatment includes controlling blood sugar, managing blood pressure, low-protein diet, and avoiding ...


'heart disease, kidney damage, eye damage, ner'

## Step 11 — Reload Vector Store Next Time

Skip Steps 5–8 next time and just run this cell.

In [22]:
# Uncomment and run this next time instead of rebuilding
# embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
# vectorstore = FAISS.load_local('faiss_index', embeddings, allow_dangerous_deserialization=True)
# print('✅ Loaded from faiss_index/')